<a href="https://colab.research.google.com/github/Avichay3/Full-training/blob/attention/attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [5]:
class SelfAttentionOneHead(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()

        self.d_model = d_model
        self.d_k = d_k

        # linear layers for q, k, v
        self.q_layer = nn.Linear(d_model, d_k, bias=False)
        self.k_layer = nn.Linear(d_model, d_k, bias=False)
        self.v_layer = nn.Linear(d_model, d_k, bias=False)

    def forward(self, x, mask=None):
        # x shape:(batch_size, seq_len, d_model)

        q = self.q_layer(x)
        k = self.k_layer(x)
        v = self.v_layer(x)

        # attention scores
        scores = torch.matmul(q, k.transpose(1, 2))

        # scale
        scores = scores / math.sqrt(self.d_k)

        # optional mask
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # turn scores into probabilities
        attn_weights = F.softmax(scores, dim=-1)

        # weighted sum of v
        out = torch.matmul(attn_weights, v)

        return out, attn_weights

In [7]:
torch.manual_seed(0)

batch_size = 2
seq_len = 4
d_model = 8
d_k = 8

x = torch.randn(batch_size, seq_len, d_model)

attn = SelfAttentionOneHead(d_model, d_k)
out, weights = attn(x)

print("x shape:", x.shape)
print("out shape:", out.shape)
print("weights shape:", weights.shape)

x shape: torch.Size([2, 4, 8])
out shape: torch.Size([2, 4, 8])
weights shape: torch.Size([2, 4, 4])


In [8]:
print(weights[0])
print()
print(weights[0].sum(dim=-1))

tensor([[0.3631, 0.1601, 0.3278, 0.1490],
        [0.2494, 0.2568, 0.2139, 0.2799],
        [0.2425, 0.2233, 0.2496, 0.2846],
        [0.3087, 0.2223, 0.2323, 0.2367]], grad_fn=<SelectBackward0>)

tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [9]:
# just checking that each row in attention really sums to 1
row_sums = weights[0].sum(dim=-1)
print(row_sums)

tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)
